# Допуск: Many-Shot In-Context Learning

**Статья:** R. Agarwal, A. Singh, L. M. Zhang, B. Bohnet, L. Rosias, S. C. Y. Chan, B. Zhang et al.,
2024 — <https://arxiv.org/abs/2404.11018>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций или классов заменены: определите, что чему
  соответствует, и ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **перепутаны местами блоки кода** — какой-то блок стоит не на своём месте: найдите его;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его;
- **вопрос по коду** — ошибок в коде нет, ответьте на вопросы по статье с опорой на код.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / рисунок / цитата) нарушено и как должно быть. Код писать не нужно — достаточно
объяснить словами. Запускать и отлаживать код тоже не нужно: все задания решаются чтением
кода и статьи.

Общие импорты:

In [ ]:
import random
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

`model`, `tok` — небольшая предобученная языковая модель и её токенизатор из `transformers`.
Сниппеты — части одной реализации; функции, которые используются, но не показаны, считайте
корректными:

- `generate(model, tok, prompt, n=1, temperature=0.0) -> list[str]` — `n` продолжений промпта
  (при `temperature=0.0` — жадное декодирование);
- `final_answer(text) -> str` — финальный ответ, извлечённый из решения;
- `sequence_logprob(model, tok, prompt, text) -> float` — log p(text | prompt) по модели.

## Задание 1. Три контекста

**Тип задания:** что есть что — названия функций/классов заменены, определите, что чему соответствует.

`shots` — список пар (задача, решение), `format_demos` — несколько фиксированных пар
(задача, решение). Функции строят контекст из примеров (тестовая задача добавляется отдельно и
здесь не показана). Определите, какая функция строит контекст для обычного many-shot ICL,
какая — для **Unsupervised ICL**, а какая — для **контрольного эксперимента из §4** (в статье
он поставлен на машинном переводе, здесь для единообразия записан в формате «задача —
решение»). Ошибок в коде нет.

In [ ]:
def context_a(shots, format_demos):
    return "".join(f"Problem: {p}\nSolution: {s}\n\n" for p, s in shots)

def context_b(shots, format_demos, base=25):
    reps = shots[:base] * (len(shots) // base)
    random.shuffle(reps)
    return "".join(f"Problem: {p}\nSolution: {s}\n\n" for p, s in reps)

def context_c(shots, format_demos):
    body = "You will be provided Problems similar to the ones below:\n"
    body += "".join(f"Problem: {p}\n" for p, _ in shots)
    body += "Now, I am going to give you a series of demonstrations of math Problems and Solutions.\n"
    body += "".join(f"Problem: {p}\nSolution: {s}\n\n" for p, s in format_demos)
    return body

**Вопросы.**
1. В одной из функций с ростом числа примеров K растёт только часть промпта. Какая? Почему,
   по гипотезе авторов (§3), такой промпт может помогать на MATH, и почему он не помогает в
   машинном переводе (§A.3)?
2. Какие два эффекта добавления примеров разделяет контрольный эксперимент? Что показал
   результат (§4.6)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Сбор примеров для Reinforced ICL

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

`problems`, `answers` — задачи обучающей выборки и их правильные финальные ответы
(человеческих решений нет). `cot_prompt(problem)` строит few-shot промпт с chain-of-thought
для задачи (корректна, не показана). Найдите одно
отклонение от статьи.

In [ ]:
def collect_reinforced_shots(model, tok, problems, answers, n_samples=10):
    shots = []
    for problem, answer in zip(problems, answers):
        prompt = cot_prompt(problem)                         # few-shot CoT prompt for this problem
        samples = generate(model, tok, prompt, n=n_samples, temperature=1.0)     # list[str]
        best = max(samples, key=lambda r: sequence_logprob(model, tok, prompt, r))
        shots.append((problem, best))
    return shots        # (problem, model-generated rationale) pairs for the many-shot prompt

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Перевёрнутые метки

**Тип задания:** перепутаны местами блоки кода.

Анализ тональности с заменёнными метками (`relabel` — `FLIPPED` или `ABSTRACT`). `shots` и
`test` — списки пар (предложение, исходная метка). Найдите одно отклонение.

In [ ]:
LABELS = ["negative", "neutral", "positive"]
FLIPPED = {"negative": "neutral", "neutral": "positive", "positive": "negative"}
ABSTRACT = {"negative": "A", "neutral": "B", "positive": "C"}

def build_prompt(shots, sentence, relabel):
    prompt = ""
    prompt += f"Sentence: {sentence}\nLabel:"
    for s, y in shots:
        prompt += f"Sentence: {s}\nLabel: {relabel[y]}\n\n"
    return prompt

@torch.no_grad()
def accuracy(model, tok, shots, test, relabel):
    options = [relabel[y] for y in LABELS]
    option_ids = [tok(" " + o).input_ids[0] for o in options]         # first token of each label
    correct = 0
    for sentence, y in test:
        x = tok(build_prompt(shots, sentence, relabel), return_tensors="pt").to(model.device)
        logits = model(**x).logits[0, -1, option_ids]                  # (3,)
        correct += options[logits.argmax()] == relabel[y]
    return correct / len(test)

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. NLL на эталонных решениях

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

`make_prompt(K, problem)` строит K-shot промпт (корректна, не показана), `test` — список пар
(задача, эталонное решение). Найдите одно отклонение от статьи.

In [ ]:
@torch.no_grad()
def reference_nll(model, tok, prompt, solution):
    p = tok(prompt).input_ids
    s = tok(solution).input_ids
    ids = torch.tensor([p + s], device=model.device)                  # (1, P + S)
    logits = model(ids).logits[0, len(p) - 1 : -1]                    # (S, V): predictions of solution tokens
    return F.cross_entropy(logits, ids[0, len(p):]).item()            # mean NLL per solution token

def nll_curve(model, tok, make_prompt, test, Ks):
    # NLL falls steadily as K grows, so task accuracy also keeps growing with K:
    # adding shots never hurts, and the NLL curve can be used to pick the best K
    return {K: np.mean([reference_nll(model, tok, make_prompt(K, q), sol) for q, sol in test])
            for K in Ks}

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Линейная классификация в высокой размерности

**Тип задания:** вопрос по коду — ошибок в коде нет, ответьте на вопросы.

Генерация задачи (по §A.5) и базовая модель для сравнения из §4.2. В промпте каждый пример
записан как `Input: <N чисел>` / `Output: <метка>`.

In [ ]:
def make_task(N, K, n_eval=25, seed=0):
    rng = np.random.default_rng(seed)
    a = rng.integers(1, 1000, size=N)                     # hidden weight vector
    t = a @ rng.integers(1, 1000, size=N)                 # threshold through a random pivot point
    def sample(k):                                        # k points per class
        pos, neg = [], []
        while len(pos) < k or len(neg) < k:
            x = rng.integers(1, 1000, size=N)
            (pos if a @ x > t else neg).append(x)
        return pos[:k] + neg[:k], ["Foo"] * k + ["Bar"] * k
    return sample(K), sample(n_eval)                      # (shots, test)

def baseline(shots, test, k=5):
    X = torch.tensor(np.array(shots[0]), dtype=torch.float)          # (2K, N)
    Q = torch.tensor(np.array(test[0]), dtype=torch.float)           # (2 n_eval, N)
    idx = torch.cdist(Q, X).topk(k, largest=False).indices           # (2 n_eval, k)
    votes = np.array(shots[1])[idx.numpy()]                          # (2 n_eval, k)
    return np.mean([max(set(v), key=list(v).count) == y for v, y in zip(votes, test[1])])

**Вопросы.**
1. Почему метки — `Foo` / `Bar`, а не, например, `positive` / `negative`? Какой другой
   эксперимент статьи проверяет то же свойство many-shot ICL?
2. Какой baseline реализован в `baseline` и почему авторы сравниваются именно с ним, а не с
   логистической регрессией, которая на линейно разделимых данных почти безошибочна? Какой
   вывод о механизме ICL авторы делают из сравнения (§4.2)?
3. Зачем порог `t` проводится через случайную точку из того же куба, а не берётся, например,
   случайным числом из [1, 1000]? Что случилось бы с `sample` во втором случае?

**Ваш ответ:**

_Напишите ответ здесь._